# Exploring ACT DR6 CMB-only Likelihood Data Products

This notebook explores the data products from the DR6 foreground-marginalized CMB likelihood.

**Data source:** `/home/jiaqu/DR6-ACT-lite/act_dr6_cmbonly/data/act_dr6_cmb_sacc.fits`

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sacc

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Load the SACC File

SACC (Save All Correlations and Covariances) stores two-point function measurements, covariance matrices, and bandpower windows.

In [2]:
sacc_file = '/home/jiaqu/DR6-ACT-lite/act_dr6_cmbonly/data/act_dr6_cmb_sacc.fits'
s = sacc.Sacc.load_fits(sacc_file)
print(f"SACC file loaded! Number of data points: {len(s.data)}")

SACC file loaded! Number of data points: 127


## 2. List Available Tracers and Data Types

In [3]:
print("TRACERS:")
for name, tracer in s.tracers.items():
    print(f"  {name}: {type(tracer).__name__}")

print("\nDATA TYPES:")
for dt in s.get_data_types():
    indices = s.indices(dt)
    print(f"  {dt}: {len(indices)} points")

TRACERS:
  dr6_cmb_s0: MiscTracer
  dr6_cmb_s2: MiscTracer

DATA TYPES:
  cl_00: 45 points
  cl_0e: 40 points
  cl_ee: 42 points


In [4]:
# Tracer combinations
print("TRACER COMBINATIONS:")
for dt in s.get_data_types():
    print(f"\n{dt}:")
    tracer_combos = set()
    for idx in s.indices(dt):
        dp = s.data[idx]
        tracer_combos.add((dp.tracers[0], dp.tracers[1]))
    for t1, t2 in sorted(tracer_combos):
        count = len(s.indices(dt, (t1, t2)))
        print(f"  {t1} x {t2}: {count} bins")

TRACER COMBINATIONS:

cl_00:
  dr6_cmb_s0 x dr6_cmb_s0: 45 bins

cl_0e:
  dr6_cmb_s0 x dr6_cmb_s2: 40 bins

cl_ee:
  dr6_cmb_s2 x dr6_cmb_s2: 42 bins


## 3. Extract and Plot Bandpowers (TT, TE, EE)

Bandpowers are stored as $D_\ell = \ell(\ell+1)C_\ell/(2\pi)$ in $\mu K^2$ (via `ell_factor=True` in the likelihood).

In [5]:
def extract_spectrum(s, data_type):
    """Extract ell, Cl, and errors for a data type."""
    indices = s.indices(data_type)
    if len(indices) == 0:
        return None, None, None
    
    ells = np.array([s.data[i].get_tag('ell') for i in indices])
    cls = np.array([s.data[i].value for i in indices])
    
    if s.covariance is not None:
        errs = np.sqrt(np.diag(s.covariance.dense[np.ix_(indices, indices)]))
    else:
        errs = None
    
    sort_idx = np.argsort(ells)
    return ells[sort_idx], cls[sort_idx], errs[sort_idx] if errs is not None else None

# Standard SACC type mappings
spectrum_map = {'TT': 'cl_00', 'TE': 'cl_0e', 'EE': 'cl_ee', 'BB': 'cl_bb'}
available_types = s.get_data_types()
print(f"Data types in file: {available_types}")

Data types in file: ['cl_00', 'cl_0e', 'cl_ee']


In [6]:
# Extract spectra
spectra = {}
for name, sacc_type in spectrum_map.items():
    if sacc_type in available_types:
        ells, cls, errs = extract_spectrum(s, sacc_type)
        if ells is not None:
            spectra[name] = {'ells': ells, 'cls': cls, 'errs': errs, 'type': sacc_type}
            print(f"{name}: {len(ells)} bins, ell=[{ells.min():.0f}, {ells.max():.0f}]")

TT: 45 bins, ell=[598, 6124]
TE: 40 bins, ell=[598, 4124]
EE: 42 bins, ell=[498, 4124]


In [ ]:
# Plot spectra
if spectra:
    n = len(spectra)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
    if n == 1: axes = [axes]
    
    for ax, (name, data) in zip(axes, spectra.items()):
        ells, cls, errs = data['ells'], data['cls'], data['errs']
        dls = cls  # SACC already stores Dℓ
        dl_errs = errs  # errors also in Dℓ units
        
        if dl_errs is not None:
            ax.errorbar(ells, dls, yerr=dl_errs, fmt='o', capsize=2, ms=4)
        else:
            ax.plot(ells, dls, 'o', ms=4)
        
        ax.set_xlabel(r'$\ell$')
        ax.set_ylabel(r'$D_\ell$ [$\mu K^2$]')
        ax.set_title(f'{name} Power Spectrum')
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Extract and Visualize the Covariance Matrix

In [ ]:
if s.covariance is not None:
    cov = s.covariance.dense
    print(f"Covariance shape: {cov.shape}")
    
    # Correlation matrix
    std = np.sqrt(np.diag(cov))
    corr = cov / np.outer(std, std)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Covariance (log scale)
    cov_plot = np.sign(cov) * np.log10(np.abs(cov) + 1e-30)
    im1 = ax1.imshow(cov_plot, cmap='RdBu_r', aspect='equal')
    ax1.set_title('Covariance (signed log scale)')
    plt.colorbar(im1, ax=ax1)
    
    # Correlation
    im2 = ax2.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='equal')
    ax2.set_title('Correlation Matrix')
    plt.colorbar(im2, ax=ax2)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Condition number: {np.linalg.cond(cov):.2e}")
else:
    print("No covariance matrix found!")

## 5. Extract Bandpower Windows

In [ ]:
print("Checking for bandpower windows...")
if hasattr(s, 'windows') and s.windows is not None:
    windows = s.windows
    print(f"Windows type: {type(windows)}")
    print(f"Attributes: {[a for a in dir(windows) if not a.startswith('_')]}")
    
    if hasattr(windows, 'values'):
        W = windows.values
        print(f"Window matrix shape: {W.shape}")
else:
    print("No windows attribute found.")
    print("\nChecking data point tags:")
    print(s.data[0].tags)

In [ ]:
# Plot windows if available
if hasattr(s, 'windows') and s.windows is not None and hasattr(s.windows, 'values'):
    W = s.windows.values
    n_bins, n_ells = W.shape if len(W.shape) > 1 else (1, len(W))
    
    # Plot a few example windows
    fig, ax = plt.subplots(figsize=(10, 6))
    indices = np.linspace(0, n_bins-1, min(5, n_bins), dtype=int)
    for idx in indices:
        ax.plot(W[idx, :] if len(W.shape) > 1 else W, label=f'Bin {idx}')
    ax.set_xlabel(r'$\ell$')
    ax.set_ylabel('Weight')
    ax.set_title('Example Bandpower Windows')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
    
    # Window matrix heatmap
    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(W, aspect='auto', cmap='viridis')
    ax.set_xlabel(r'Theory $\ell$')
    ax.set_ylabel('Bin index')
    ax.set_title('Bandpower Window Matrix')
    plt.colorbar(im, ax=ax)
    plt.show()

## 6. Summary for Lensing Likelihood

### Data Products

| Product | Description | Lensing Use |
|---------|-------------|-------------|
| **Bandpowers** | TT, TE, EE spectra | Constrain CMB entering lensing normalization |
| **Covariance** | Full cross-spectrum covariance | Joint CMB+lensing analysis |
| **Windows** | Theory-to-bandpower mapping | Bin theory predictions |

### Lensing Corrections

1. **Normalization (dAL/dC)**: Lensing normalization depends on CMB power spectra
2. **N1 Bias**: Connected 4-point contribution depends on TT, TE, EE
3. **CMB-marginalized covariance**: For `lens_only=True` mode

In [ ]:
print("SUMMARY")
print(f"File: {sacc_file}")
print(f"Total points: {len(s.data)}")
print(f"Tracers: {list(s.tracers.keys())}")
print(f"Data types: {s.get_data_types()}")
print(f"Covariance: {s.covariance.dense.shape if s.covariance else 'None'}")
for name, data in spectra.items():
    print(f"{name}: {len(data['ells'])} bins, ell=[{data['ells'].min():.0f}, {data['ells'].max():.0f}]")